# 01 — Data Exploration

This notebook explores the customer-order dataset used in the **Quick-Commerce Dark Store Network Analysis — Bengaluru Case Study**.

**Goal:** understand the structure, quality, demand timing, order values, and geographic spread of the customer-order data before applying K-Means clustering.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

DATA_PATH = "../data/customer_orders.csv"

df = pd.read_csv(DATA_PATH)
df["Order_Timestamp"] = pd.to_datetime(df["Order_Timestamp"], errors="coerce")

df.head()

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nDuplicate rows:", df.duplicated().sum())
print("\nDuplicate Order_IDs:", df["Order_ID"].duplicated().sum())

In [ ]:
# Basic descriptive statistics
df[["Customer_Lat", "Customer_Lon", "Order_Value_INR", "Order_Hour"]].describe()

## Demand by Hour

This view helps identify when customer demand is highest. The hourly demand pattern is useful later for operational and fulfillment planning.

In [ ]:
hourly_orders = (
    df.groupby("Order_Hour")["Order_ID"]
      .count()
      .reset_index(name="Order_Count")
      .sort_values("Order_Hour")
)

hourly_orders

In [ ]:
peak_hour = hourly_orders.loc[hourly_orders["Order_Count"].idxmax(), "Order_Hour"]
peak_orders = hourly_orders["Order_Count"].max()

print(f"Peak order hour: {int(peak_hour)}:00")
print(f"Orders at peak hour: {int(peak_orders)}")

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(hourly_orders["Order_Hour"], hourly_orders["Order_Count"], marker="o")
plt.xlabel("Order Hour")
plt.ylabel("Number of Orders")
plt.title("Customer Orders by Hour")
plt.xticks(range(int(hourly_orders["Order_Hour"].min()), int(hourly_orders["Order_Hour"].max()) + 1))
plt.grid(alpha=0.3)
plt.show()

## Order Value Analysis

In [ ]:
order_value_summary = {
    "Total Order Value (INR)": df["Order_Value_INR"].sum(),
    "Average Order Value (INR)": df["Order_Value_INR"].mean(),
    "Median Order Value (INR)": df["Order_Value_INR"].median(),
    "Minimum Order Value (INR)": df["Order_Value_INR"].min(),
    "Maximum Order Value (INR)": df["Order_Value_INR"].max(),
}
pd.Series(order_value_summary)

In [ ]:
hourly_value = (
    df.groupby("Order_Hour")["Order_Value_INR"]
      .agg(["sum", "mean"])
      .reset_index()
      .rename(columns={"sum": "Total_Order_Value", "mean": "Average_Order_Value"})
)

hourly_value

## Geographic Spread

The project uses customer latitude and longitude as the geographic basis for K-Means clustering. This section checks the overall geographic range and displays individual customer orders.

In [ ]:
print("Latitude range:", df["Customer_Lat"].min(), "to", df["Customer_Lat"].max())
print("Longitude range:", df["Customer_Lon"].min(), "to", df["Customer_Lon"].max())

In [ ]:
plt.figure(figsize=(8, 8))
plt.scatter(df["Customer_Lon"], df["Customer_Lat"], s=8, alpha=0.35)
plt.xlabel("Customer Longitude")
plt.ylabel("Customer Latitude")
plt.title("Geographic Distribution of Customer Orders")
plt.grid(alpha=0.2)
plt.show()

## Initial Findings

Use the outputs above to record the main data observations before moving to clustering.

Typical observations to capture:

- Number of customer orders
- Peak order hour
- Average order value
- Geographic spread of orders
- Whether missing values or duplicate order IDs need attention

These observations are descriptive only; they do not identify dark-store locations by themselves.